In [3]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,  DataLoader, random_split
import random


In [4]:
dataset = load_dataset("Anthropic/hh-rlhf")

In [5]:
subset_size = 15000

train_subset = dataset["train"].select(range(subset_size))

chosen_texts = [sample["chosen"] for sample in train_subset]
rejected_texts = [sample["rejected"] for sample in train_subset]


In [6]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
chosen_embeddings = embedding_model.encode(
    chosen_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True)

rejected_embeddings = embedding_model.encode(
    rejected_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True)

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

#гарантируем существования папки с embeddings и сохраняем embeddings

In [15]:
os.makedirs("../data/embeddings", exist_ok=True)

np.save("../data/embeddings/chosen_15k.npy", chosen_embeddings)
np.save("../data/embeddings/rejected_15k.npy", rejected_embeddings)

print("Embeddings saved successfully.")
print(f"Chosen shape: {chosen_embeddings.shape}")
print(f"Rejected shape: {rejected_embeddings.shape}")
print(f"Expected file size per array: ~{chosen_embeddings.nbytes / 1024**2:.1f} MB")

Embeddings saved successfully.
Chosen shape: (15000, 384)
Rejected shape: (15000, 384)
Expected file size per array: ~22.0 MB


##RewardDataSet + DataLoader

In [8]:
class RewardDataSetAugmented(Dataset):
    def __init__(self, chosen_path, rejected_path):
        self.chosen = np.load(chosen_path)
        self.rejected = np.load(rejected_path)
        assert len(self.chosen) == len(self.rejected), "Несоответствие в длине набора данных!"
    
    def __len__(self):
        return len(self.chosen)
    
    def __getitem__(self, index):
        c = self.chosen[index]
        r = self.rejected[index]

        if random.random() > 0.5:
            first, second = c, r
            label = 1.0
        else:
            first, second = r, c
            label = 0.0
        

        #формирование признаков
        #dim = 768*5 + 1 = 3841
        diff = first - second
        abs_diff = np.abs(diff)
        prod = first * second

        norm_f = np.linalg.norm(first)
        norm_s = np.linalg.norm(second)
        cos_sim = np.dot(first, second) / (norm_f * norm_s + 1e-8)

        features = np.concatenate([first, second, diff, abs_diff, prod, [cos_sim]])
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

#Инициализация и сплит

full_dataset = RewardDataSetAugmented("../data/embeddings/chosen_15k.npy", "../data/embeddings/rejected_15k.npy")
print(len(full_dataset))
train_size = int(0.8*len(full_dataset))
val_size = len(full_dataset) - train_size
train_subset, val_subset = random_split(
    full_dataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_subset, batch_size=256, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=512, shuffle=False, num_workers=0)

print(f"Dataset готов. Тренировочный: {len(train_subset)} | Валидационный: {len(val_subset)}")



15000
Dataset готов. Тренировочный: 12000 | Валидационный: 3000


##RewardMLP

In [ ]:
class RewardMLP(nn.Module):
    def __init__(self, input_dim=3841):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128,64), 
            nn.ReLU(), 
            nn.Dropout(0.2),
            nn.Linear(64,1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = RewardMLP().to(device)

print(f"Device: {device} | Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: mps | Model parameters: 254,337


##Training Loop + Валидация + Сохранение

In [36]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

epochs = 30
best_val_loss = float('inf')
patience = 5
patience_counter = 0
best_val_acc = 0.0

os.makedirs("../models", exist_ok=True)

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * features.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        train_correct += (preds==labels).sum().item()
        train_total += features.size(0)

    #Валидация
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * features.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += features.size(0)
    
    train_loss /= train_total
    val_loss /= val_total
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "../models/reward_mlp_15k.pt")
        print(f"Saved best model. Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}. Best Val Acc: {best_val_acc:.4f}")
            break


Epoch 1/30 | Train Loss: 0.6923 Acc: 0.5172 | Val Loss: 0.6893 Acc: 0.5650
Saved best model. Val Loss: 0.6893 | Val Acc: 0.5650
Epoch 2/30 | Train Loss: 0.6846 Acc: 0.5771 | Val Loss: 0.6778 Acc: 0.5780
Saved best model. Val Loss: 0.6778 | Val Acc: 0.5780
Epoch 3/30 | Train Loss: 0.6721 Acc: 0.5786 | Val Loss: 0.6676 Acc: 0.5867
Saved best model. Val Loss: 0.6676 | Val Acc: 0.5867
Epoch 4/30 | Train Loss: 0.6631 Acc: 0.5965 | Val Loss: 0.6561 Acc: 0.5997
Saved best model. Val Loss: 0.6561 | Val Acc: 0.5997
Epoch 5/30 | Train Loss: 0.6588 Acc: 0.5992 | Val Loss: 0.6555 Acc: 0.6050
Saved best model. Val Loss: 0.6555 | Val Acc: 0.6050
Epoch 6/30 | Train Loss: 0.6564 Acc: 0.6020 | Val Loss: 0.6510 Acc: 0.6017
Saved best model. Val Loss: 0.6510 | Val Acc: 0.6017
Epoch 7/30 | Train Loss: 0.6547 Acc: 0.6028 | Val Loss: 0.6499 Acc: 0.6080
Saved best model. Val Loss: 0.6499 | Val Acc: 0.6080
Epoch 8/30 | Train Loss: 0.6520 Acc: 0.6080 | Val Loss: 0.6482 Acc: 0.6130
Saved best model. Val Loss: 0